In [1]:
import sys
import pickle
from tqdm import tqdm

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger, ReaderMetrics
from src.utils.data_structs import TripletCreator

/home/dzigen/Desktop/PersonalAI/pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [3]:
#PKL_GRAPH_PATH = 'C:/Users/nikit/temp_files/pickled_graphs/DiaasqGigachat.pickle'
PKL_GRAPH_PATH = '../../data/pickled_graphs/DiaasqGigachat.pickle'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)
print(len(formated_triplets))

211542


#### 2. Задаём конфигурацию графа знаний

In [4]:
# in-memory storage
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)

In [5]:
#
LANGUAGE = 'en'

# 
RETRIEVER_CONFIG = KnowledgeRetrieverConfig(
    retriever_method='astar',
    retriever_config=AStarGraphSearchConfig(),
    cache_config=KV_STORAGE_CONFIG)

# embedder hyperp
DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

# vector dbs hyperp
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

In [6]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=NODES_DB_PATH, db_name='vectorized_nodes', need_to_clear=NEED_TO_CLEAR)),
        tripletsdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=TRIPLETS_DB_PATH, db_name='vectorized_triplets', need_to_clear=NEED_TO_CLEAR)),
        embedder_config=EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=RETRIEVER_CONFIG,
        answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE)),
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(lang=LANGUAGE),
        updator_config=LLMUpdatorConfig(lang=LANGUAGE)),
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [7]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


#### 4. Добавляем в граф загруженные триплеты

In [8]:
print("uploading data to graph-storage")
rkg_main.kg_model.graph_struct.create_triplets(formated_triplets[:1000])
print("uploading data to vector-storage")
rkg_main.kg_model.embeddings_struct.add_triplets(formated_triplets[:1000])

uploading data to graph-storage


100%|██████████| 1000/1000 [00:00<00:00, 131656.22it/s]


uploading data to vector-storage


100%|██████████| 8/8 [00:11<00:00,  1.45s/it]


#### 5. Q&A

In [8]:
METRICS = ReaderMetrics(base_dir="../..", bs_model_path="google/electra-base-discriminator")
# METRICS.exact_match(gen_answers, trgt_answers)

Loading Meteor...
Loading ExactMatch
Loading BertScore


In [43]:
N = 1000
QUESTION_NUM = 20
episodic_text = []
for i in range(N):
    if formated_triplets[i].end_node.type.value == 'episodic':
        episodic_text.append(formated_triplets[i].end_node.name)
unique_episodic_texts = list(set(episodic_text))
print(len(episodic_text), len(unique_episodic_texts))

llm_agent = GigaChatAgent()

QUESTION_GEN_PROMPT = "Generate one question based on given dialogue below. Generate question in English.\n\nDialogue:\n{d}\n\nQuestion:\n"
ANSWER_GEN_PROMPT = "Generate answer for the question based on given dialogue below. Generate answer in English. Answer needs to be generated in a short format: in several phrases; dont generate full sentence as an answer.\n\nDialogue:\n{d}\n\nQuestion:{q}\n\nAnswer:\n"

qa_examples = []
for text in tqdm(unique_episodic_texts[:QUESTION_NUM]):
    question = llm_agent.generate(user_prompt=QUESTION_GEN_PROMPT.format(d=text)).strip()
    answer = llm_agent.generate(user_prompt=ANSWER_GEN_PROMPT.format(d=text, q=question)).strip()
    print("Q: ", question)
    print("A: ", answer)
    qa_examples.append((question, answer))

  6%|▌         | 1/18 [00:03<00:54,  3.22s/it]

Q:  What mobile phone does Fiona's family have that doesn't get hot when playing games during cold nights?
A:  Fiona's family has a mobile phone that doesn't get hot when playing games during cold nights.


 11%|█         | 2/18 [00:06<00:49,  3.11s/it]

Q:  What is Lauren's opinion about the 13promax battery compared to other batteries in terms of battery life?
A:  Lauren thinks the 13promax battery has better battery life than other batteries in mobile phones.


 17%|█▋        | 3/18 [00:10<00:56,  3.78s/it]

...

Q:  What are the differences between the heat dissipation performance of IQOO9 and Xiaomi Mi 12Pro, according to Gregory?
A:  According to Gregory, the heat dissipation performance of IQOO9 is superior to Xiaomi Mi 12Pro.


 94%|█████████▍| 17/18 [01:10<00:03,  3.84s/it]

Q:  What are some issues users have experienced with their mobile phones regarding signal quality, battery life, and performance?
A:  Users have experienced issues with poor signal quality, fast battery drain, and performance problems on their mobile phones.


100%|██████████| 18/18 [01:14<00:00,  4.13s/it]

Q:  Why does Bailey think that the battery life of Apple phones is still weak even after 21 years?
A:  Bailey thinks the battery life of Apple phones is still weak after 21 years because some people say it's not their weakness, and from his experience using his sister's Apple phone.


In [10]:
qa_examples = [
  ("What is Lauren's opinion about the 13promax battery compared to other batteries in terms of battery life?",
  'Lauren thinks the 13promax battery has better battery life than other batteries in mobile phones.'),
  ("What does Emily think about the performance of her brother's GT2PRO and IQOO7 smartphones during voice calls.",
  'Emily thinks GT2PRO is not as hot during voice calls compared to IQOO7.'),
  ("What is the difference between the heat dissipation performance of the IQOO9 and Xiaomi Mi 12Pro smartphones according to Laura's experience?",
  'Laura finds the heat dissipation performance of the IQOO9 superior to the Xiaomi Mi 12Pro, resulting in less heating while using it for long periods.'),
  ("What are the reasons behind Rodrigo's dislike for Xiaomi's MIUI software, specifically mentioning the frequent bugs he has experienced over the past two years?",
  "Rodrigo dislikes Xiaomi's MIUI software due to frequent bugs he has experienced over the past two years, including small bugs that affect usage, such as force restarts."),
  ("What is Margaret's main concern regarding her MIX4 smartphone?",
  "problem with the photography camera module and its poor performance."),
  ('What are the advantages of the Xiaomi Mi 10pro mentioned by Horace and Jesse, especially in comparison to other devices?',
  'Advantages of Xiaomi Mi 10pro: splitscreen capability, fast charging, screen durability, waterproofing, good signal strength, fast gaming performance. Comparatively better than other devices (especially Apple) in terms of battery life and signal strength.'),
  ('What issues have James and Hailey experienced with their MIX4 devices, especially regarding the photography camera module?',
  "James experienced multiple crashes in a short period, while Hailey mentioned her device being used many times."),
  ("What was Kevin's experience with his brother's GT2PRO and Xiaomi 10Pro?",
  "Kevin found his brother's GT2PRO less hot than expected during voice calls, while Xiaomi 10Pro didn't heat up when recording videos."),
  ("What mobile phone does Kayla's brother have that doesn't get hot during voice calls, according to her?",
  "IQOO7")]

In [11]:
rkg_main.answer_question(qa_examples[-2][0])

"Kevin's experiences with his brother's GT2PRO and Xiaomi 10Pro included observations about the devices' relative performance compared to Apple products, particularly regarding signal strength and battery life. He expressed concern about signal issues with Xiaomi and hesitation in switching to Apple due to signal and electrical issues."